# Extracted functions

This notebook contains the reusable imports, constants, helper functions, classes, plotting functions, prediction utilities, and correction/evaluation utilities extracted from the original pipeline notebook.

Execution/training/evaluation cells that immediately run experiments were intentionally excluded.


In [ ]:
import os
from pathlib import Path
import math
import time
import random
import json
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

from PIL import Image, ImageOps
from scipy.ndimage import gaussian_filter

import numpy as np
import pydicom
import pandas as pd

from scipy.ndimage import gaussian_filter


In [ ]:
def _first_existing(cols, candidates):
    for c in candidates:
        if c in cols:
            return c
    return None

def _resolve_external_path(path_value, image_root=EXTERNAL_IMAGE_ROOT):
    p = Path(str(path_value))
    if p.is_absolute():
        return str(p)
    return str(Path(image_root) / p)

def _load_grayscale_image_for_external(path):
    """Load PNG/JPG/TIFF with PIL; fall back to DICOM via pydicom when needed."""
    path = str(path)
    try:
        with Image.open(path) as im:
            return ImageOps.grayscale(im)
    except Exception as pil_error:
        suffix = Path(path).suffix.lower()
        if suffix in {'.dcm', '.dicom', ''}:
            try:
                ds = pydicom.dcmread(path)
                arr = ds.pixel_array.astype(np.float32)
                arr -= np.nanmin(arr)
                mx = np.nanmax(arr)
                if mx > 0:
                    arr /= mx
                arr = (arr * 255.0).clip(0, 255).astype(np.uint8)
                return Image.fromarray(arr, mode='L')
            except Exception as dicom_error:
                raise RuntimeError(f'Could not load external image as PIL or DICOM: {path}\nPIL error: {pil_error}\nDICOM error: {dicom_error}')
        raise RuntimeError(f'Could not load external image with PIL: {path}\nError: {pil_error}')

def build_external_localization_df(csv_path, image_root, source_name, levels, compute_image_size):
    """
    Convert df_coords.csv from Localization_lumbar_spine_aug into a data_merged-like
    long dataframe suitable for the conditional localization training workflow.

    Required after standardization:
      img_path, x_norm, y_norm, level
    """
    csv_path = Path(csv_path)
    if not csv_path.exists():
        print(f'External coordinate CSV not found, skipping: {csv_path}')
        return pd.DataFrame()
    df = pd.read_csv(csv_path).copy()
    if 'exists' in df.columns:
        df = df[df['exists'].astype(bool)].copy()
    path_col = _first_existing(df.columns, ['full_path', 'path', 'image_path', 'filepath', 'file_path', 'filename', 'img_path'])
    if path_col is None:
        raise ValueError(f'External CSV has no path column. Available columns: {list(df.columns)}')
    if {'relative_x', 'relative_y'}.issubset(df.columns):
        x_norm = df['relative_x'].astype(float)
        y_norm = df['relative_y'].astype(float)
    elif {'x_norm', 'y_norm'}.issubset(df.columns):
        x_norm = df['x_norm'].astype(float)
        y_norm = df['y_norm'].astype(float)
    else:
        x_col = _first_existing(df.columns, ['x', 'coord_x', 'center_x', 'px', 'column'])
        y_col = _first_existing(df.columns, ['y', 'coord_y', 'center_y', 'py', 'row'])
        w_col = _first_existing(df.columns, ['image_width', 'width', 'img_width', 'W'])
        h_col = _first_existing(df.columns, ['image_height', 'height', 'img_height', 'H'])
        if x_col is None or y_col is None:
            raise ValueError('External CSV needs relative_x/relative_y, x_norm/y_norm, or x/y-like columns.')
        if w_col is None or h_col is None:
            raise ValueError('Absolute external coordinates require image_width/image_height columns.')
        x_norm = df[x_col].astype(float) / np.maximum(df[w_col].astype(float) - 1, 1)
        y_norm = df[y_col].astype(float) / np.maximum(df[h_col].astype(float) - 1, 1)
    level_col = _first_existing(df.columns, ['level', 'vertebra', 'disc_level'])
    if level_col is None:
        raise ValueError(f'External CSV has no level column. Available columns: {list(df.columns)}')
    out = pd.DataFrame({'img_path': df[path_col].apply(lambda p: _resolve_external_path(p, image_root=image_root)), 'level': df[level_col].astype(str), 'x_norm': np.clip(x_norm.astype(float), 0.0, 1.0), 'y_norm': np.clip(y_norm.astype(float), 0.0, 1.0)})
    out = out[out['level'].isin(levels)].copy()
    out = out.dropna(subset=['img_path', 'level', 'x_norm', 'y_norm']).reset_index(drop=True)
    out['external_image_id'] = out['img_path'].apply(lambda p: Path(p).stem)
    if 'study_id' in df.columns:
        raw_study = df.loc[out.index, 'study_id'].astype(str).values if len(out) <= len(df) else out['external_image_id'].values
        out['original_study_id'] = raw_study
    out['study_id'] = source_name + '_' + out['external_image_id'].astype(str)
    out['series_id'] = source_name + '_single_slice'
    out['instance_number'] = 0
    out['condition'] = 'Spinal Canal Stenosis'
    out['series_description'] = 'External single-slice lumbar localization'
    out['severity'] = 'external_unlabeled'
    out['source'] = source_name
    out['level_idx'] = out['level'].map({lvl: i for i, lvl in enumerate(levels)}).astype(int)
    out['row_id'] = out['source'].astype(str) + '_' + out['external_image_id'].astype(str) + '_' + out['level'].str.lower().str.replace('/', '_', regex=False)
    if compute_image_size:
        size_cache = {}
        for p in out['img_path'].drop_duplicates():
            try:
                im = _load_grayscale_image_for_external(p)
                size_cache[p] = im.size
            except Exception as e:
                print(f'Warning: could not read external image size for {p}: {e}')
                size_cache[p] = (np.nan, np.nan)
        out['image_width'] = out['img_path'].map(lambda p: size_cache[p][0])
        out['image_height'] = out['img_path'].map(lambda p: size_cache[p][1])
    else:
        out['image_width'] = np.nan
        out['image_height'] = np.nan
    out['x'] = out['x_norm'] * (out['image_width'].fillna(IMG_SIZE) - 1)
    out['y'] = out['y_norm'] * (out['image_height'].fillna(IMG_SIZE) - 1)
    keep_cols = ['row_id', 'study_id', 'series_id', 'condition', 'level', 'series_description', 'instance_number', 'x', 'y', 'severity', 'img_path', 'image_height', 'image_width', 'x_norm', 'y_norm', 'source', 'level_idx', 'external_image_id']
    keep_cols = [c for c in keep_cols if c in out.columns]
    out = out[keep_cols].reset_index(drop=True)
    print(f'External localization rows: {len(out):,}')
    print(f"External unique images: {out['img_path'].nunique():,}")
    print('External level counts:')
    display(out['level'].value_counts().reindex(levels))
    per_image_levels = out.groupby('img_path')['level'].nunique()
    print('External number of annotated levels per image:')
    display(per_image_levels.value_counts().sort_index())
    missing_files = [p for p in out['img_path'].drop_duplicates().head(20) if not Path(p).exists()]
    if missing_files:
        print('Warning: some external image paths do not exist. First missing paths:')
        for p in missing_files[:5]:
            print('  ', p)
    return out

def make_single_point_heatmap(img_size, x, y, sigma):
    yy = np.arange(img_size, dtype=np.float32)[:, None]
    xx = np.arange(img_size, dtype=np.float32)[None, :]
    heatmap = np.exp(-((xx - x) ** 2 + (yy - y) ** 2) / (2.0 * float(sigma) ** 2))
    return heatmap.astype(np.float32)

class ExternalReplicatedSliceLocalizationDataset(Dataset):
    """
    Dataset for external single-slice lumbar coordinate rows.

    Returns the same keys used by the localization training loop:
      image:    FloatTensor [NUM_SLICES, IMG_SIZE, IMG_SIZE]
      target:   FloatTensor [1, IMG_SIZE, IMG_SIZE]
      point_xy: FloatTensor [2]
      level_idx LongTensor scalar

    The image is a single grayscale PNG/JPG/DICOM slice replicated over all channels.
    """

    def __init__(self, df, config, levels, use_intensity_aug):
        self.df = df.reset_index(drop=True).copy()
        self.config = config
        self.cfg = config
        self.levels = list(levels)
        self.level_to_idx = {lvl: i for i, lvl in enumerate(self.levels)}
        self.img_size = int(config.img_size)
        self.num_slices = int(config.num_slices)
        self.sigma = float(config.sigma)
        self.use_intensity_aug = bool(use_intensity_aug)
        if 'level_idx' not in self.df.columns:
            self.df['level_idx'] = self.df['level'].map(self.level_to_idx).astype(int)

    def set_sigma(self, sigma):
        self.sigma = float(sigma)
        if hasattr(self, 'cfg'):
            try:
                self.cfg.sigma = float(sigma)
            except Exception:
                pass

    def __len__(self):
        return len(self.df)

    def _load_image_tensor(self, path):
        im = _load_grayscale_image_for_external(path)
        im = im.resize((self.img_size, self.img_size), resample=Image.BILINEAR)
        arr = np.asarray(im, dtype=np.float32) / 255.0
        if self.use_intensity_aug:
            if random.random() < 0.5:
                arr = np.clip(arr * random.uniform(0.85, 1.15), 0.0, 1.0)
            if random.random() < 0.25:
                arr = np.clip(arr + np.random.normal(0.0, 0.015, size=arr.shape).astype(np.float32), 0.0, 1.0)
        arr = (arr - 0.5) / 0.5
        image_1 = torch.from_numpy(arr).float().unsqueeze(0)
        return image_1.repeat(self.num_slices, 1, 1)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        x = float(np.clip(row['x_norm'], 0.0, 1.0)) * (self.img_size - 1)
        y = float(np.clip(row['y_norm'], 0.0, 1.0)) * (self.img_size - 1)
        level_idx = int(row['level_idx'])
        image = self._load_image_tensor(row['img_path'])
        target = make_single_point_heatmap(self.img_size, x, y, self.sigma)
        return {'image': image, 'target': torch.from_numpy(target).float().unsqueeze(0), 'point_xy': torch.tensor([x, y], dtype=torch.float32), 'level_idx': torch.tensor(level_idx, dtype=torch.long)}

class MixedLocalizationDataset(Dataset):
    """Concatenate datasets while preserving .cfg, combined .df, and sigma propagation."""

    def __init__(self, datasets):
        self.datasets = [d for d in datasets if d is not None and len(d) > 0]
        self.lengths = [len(d) for d in self.datasets]
        self.cum_lengths = np.cumsum(self.lengths).tolist()
        self.cfg = None
        for d in self.datasets:
            if hasattr(d, 'cfg'):
                self.cfg = d.cfg
                break
            if hasattr(d, 'config'):
                self.cfg = d.config
                break
        dfs = []
        for d in self.datasets:
            if hasattr(d, 'df'):
                dfs.append(d.df.copy())
        self.df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

    def __len__(self):
        return int(sum(self.lengths))

    def set_sigma(self, sigma):
        sigma = float(sigma)
        if self.cfg is not None:
            try:
                self.cfg.sigma = sigma
            except Exception:
                pass
        for d in self.datasets:
            if hasattr(d, 'set_sigma'):
                d.set_sigma(sigma)
            elif hasattr(d, 'sigma'):
                d.sigma = sigma

    def __getitem__(self, idx):
        if idx < 0:
            idx = len(self) + idx
        ds_start = 0
        for ds, end in zip(self.datasets, self.cum_lengths):
            if idx < end:
                sample = ds[idx - ds_start]
                return {'image': sample['image'], 'target': sample['target'], 'point_xy': sample['point_xy'], 'level_idx': sample['level_idx']}
            ds_start = end
        raise IndexError(idx)


### Original code cell 13

In [ ]:
def dataloader_worker_kwargs(num_workers=NUM_WORKERS):
    kwargs = {
        "num_workers": NUM_WORKERS,
        "pin_memory": PIN_MEMORY,
    }

    if num_workers > 0:
        kwargs["persistent_workers"] = PERSISTENT_WORKERS
        kwargs["prefetch_factor"] = PREFETCH_FACTOR

    return kwargs


### Original code cell 15

In [ ]:
def build_relative_coordinate_prior_maps_from_norm_cols(
    train_df,
    img_size=IMG_SIZE,
    sigma=PRIOR_SIGMA,
    levels=LEVELS,
    eps=PRIOR_EPS,
):
    """
    Build one prior map per level from already existing normalized coordinates:
      - x_norm
      - y_norm

    Expected columns in train_df:
      - level
      - optionally level_idx
      - x_norm
      - y_norm
    """

    priors = np.zeros((len(levels), img_size, img_size), dtype=np.float32)
    counts = np.zeros(len(levels), dtype=np.int64)

    for _, row in train_df.iterrows():
        if pd.isna(row["x_norm"]) or pd.isna(row["y_norm"]):
            continue

        if "level_idx" in train_df.columns and not pd.isna(row["level_idx"]):
            level_idx = int(row["level_idx"])
        else:
            level_idx = levels.index(str(row["level"]))

        x_rel = float(row["x_norm"])
        y_rel = float(row["y_norm"])

        x_rel = np.clip(x_rel, 0.0, 1.0)
        y_rel = np.clip(y_rel, 0.0, 1.0)

        xi = int(round(x_rel * (img_size - 1)))
        yi = int(round(y_rel * (img_size - 1)))

        priors[level_idx, yi, xi] += 1.0
        counts[level_idx] += 1

    for level_idx in range(len(levels)):
        if counts[level_idx] == 0:
            priors[level_idx] = np.ones((img_size, img_size), dtype=np.float32)
        else:
            priors[level_idx] = gaussian_filter(
                priors[level_idx],
                sigma=sigma,
                mode="constant",
            )

            mx = priors[level_idx].max()
            if mx > eps:
                priors[level_idx] /= mx

            priors[level_idx] = np.clip(priors[level_idx], eps, 1.0)

    print("Built relative-coordinate priors:", priors.shape)
    print("Counts per level:", dict(zip(levels, counts.tolist())))

    return priors


### Original code cell 17

In [ ]:
import matplotlib.pyplot as plt

def plot_relative_coordinate_scatter_from_norm_cols(train_df, levels, alpha, figsize):
    coords_by_level = {i: [] for i in range(len(levels))}
    for _, row in train_df.iterrows():
        if pd.isna(row['x_norm']) or pd.isna(row['y_norm']):
            continue
        if 'level_idx' in train_df.columns and (not pd.isna(row['level_idx'])):
            level_idx = int(row['level_idx'])
        else:
            level_idx = levels.index(str(row['level']))
        x_rel = float(row['x_norm'])
        y_rel = float(row['y_norm'])
        x_rel = np.clip(x_rel, 0.0, 1.0)
        y_rel = np.clip(y_rel, 0.0, 1.0)
        coords_by_level[level_idx].append((x_rel, y_rel))
    plt.figure(figsize=figsize)
    for level_idx, level_name in enumerate(levels):
        pts = np.asarray(coords_by_level[level_idx], dtype=np.float32)
        if len(pts) == 0:
            continue
        plt.scatter(pts[:, 0], pts[:, 1], s=12, alpha=alpha, label=f'{level_name} (n={len(pts)})')
    plt.gca().invert_yaxis()
    plt.xlim(0, 1)
    plt.ylim(1, 0)
    plt.xlabel('x_norm')
    plt.ylabel('y_norm')
    plt.title('Relative coordinate scatter by level')
    plt.legend()
    plt.grid(True, alpha=0.25)
    plt.show()


### Original code cell 19

In [ ]:
def plot_prior_maps(prior_maps, levels=LEVELS):
    fig, axes = plt.subplots(1, len(levels), figsize=(4 * len(levels), 4))

    if len(levels) == 1:
        axes = [axes]

    for i, level_name in enumerate(levels):
        axes[i].imshow(prior_maps[i], cmap="magma")
        axes[i].set_title(level_name)
        axes[i].axis("off")

    plt.tight_layout()
    plt.show()


### Original code cell 23

### Original code cell 25

In [ ]:
def move_batch_to_device(batch, device):
    out = {}
    for k, v in batch.items():
        out[k] = v.to(device, non_blocking=True) if torch.is_tensor(v) else v
    return out


def get_condition_idx_from_batch(batch):
    """
    Keeps the old model API: model(image, level_idx).

    For subarticular stenosis, level_idx may actually mean:
        0 = Left Subarticular Stenosis
        1 = Right Subarticular Stenosis

    Priority:
        1. Use batch['level_idx'] if it already exists.
        2. Otherwise use batch['side_idx'].
    """
    if "level_idx" in batch:
        return batch["level_idx"]

    if "side_idx" in batch:
        return batch["side_idx"]

    raise KeyError(
        "Batch must contain either 'level_idx' or 'side_idx'. "
        "For subarticular stenosis, reuse 'level_idx' as left/right condition index."
    )


def set_dynamic_sigma_for_epoch(dataset, epoch: int, schedule):
    if schedule is None:
        return

    sigma = None
    for start_epoch in sorted(schedule):
        if epoch >= start_epoch:
            sigma = schedule[start_epoch]

    if sigma is not None and hasattr(dataset, "set_sigma"):
        dataset.set_sigma(float(sigma))


def run_one_epoch(model, loader, optimizer, device):
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0
    total_n = 0
    loss_parts = defaultdict(float)

    pbar = tqdm(loader, leave=False, desc="train" if is_train else "val")

    for batch in pbar:
        batch = move_batch_to_device(batch, device)

        # This is the only important compatibility layer.
        # It lets the model keep using the argument name `level_idx`,
        # while the value can represent either anatomical level or left/right side.
        level_idx = get_condition_idx_from_batch(batch)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_train):
            with torch.amp.autocast("cuda", enabled=device.type == "cuda"):
                logits = model(batch["image"], level_idx)
                loss, parts = compute_loss(logits, batch)

            if is_train:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()

        bs = batch["image"].size(0)
        total_loss += float(loss.item()) * bs
        total_n += bs

        for k, v in parts.items():
            loss_parts[k] += float(v) * bs

        pbar.set_postfix(loss=total_loss / max(total_n, 1))

    logs = {"loss": total_loss / max(total_n, 1)}
    logs.update({k: v / max(total_n, 1) for k, v in loss_parts.items()})

    return logs


def train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    epochs,
    patience,
    output_dir,
    dynamic_sigma,
    n_levels=5,
):
    best_val = float("inf")
    best_path = output_dir / f"best_{MODEL_NAME}_{EXPERIMENT_DATASET}.pt"
    history = []
    bad_epochs = 0

    for epoch in range(epochs):
        set_dynamic_sigma_for_epoch(train_loader.dataset, epoch, dynamic_sigma)

        t0 = time.time()

        train_logs = run_one_epoch(
            model,
            train_loader,
            optimizer=optimizer,
            device=device,
        )

        val_logs = run_one_epoch(
            model,
            val_loader,
            optimizer=None,
            device=device,
        )

        dt = time.time() - t0

        row = {
            "epoch": epoch,
            "train_loss": train_logs["loss"],
            "val_loss": val_logs["loss"],
            "train_heatmap_loss": train_logs.get("heatmap_loss", np.nan),
            "val_heatmap_loss": val_logs.get("heatmap_loss", np.nan),
            "train_xy_loss": train_logs.get("xy_loss", np.nan),
            "val_xy_loss": val_logs.get("xy_loss", np.nan),
            "seconds": dt,
            "sigma": train_loader.dataset.cfg.sigma,
            "n_levels": n_levels,
        }

        history.append(row)

        print(
            f"Epoch {epoch + 1:03d}/{epochs} | "
            f"train {row['train_loss']:.5f} | "
            f"val {row['val_loss']:.5f} | "
            f"sigma {row['sigma']:.2f} | "
            f"n_levels {n_levels} | "
            f"{dt:.1f}s"
        )

        if row["val_loss"] < best_val:
            best_val = row["val_loss"]
            bad_epochs = 0

            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "model_name": MODEL_NAME,
                    "dataset_variant": EXPERIMENT_DATASET,
                    "img_size": IMG_SIZE,
                    "num_slices": NUM_SLICES,
                    "n_levels": n_levels,
                    "levels": LEVELS[:n_levels] if len(LEVELS) >= n_levels else LEVELS,
                    "config": row,
                },
                best_path,
            )

            print("  saved:", best_path)

        else:
            bad_epochs += 1

            if bad_epochs >= patience:
                print("Early stopping.")
                break

    if best_path.exists():
        ckpt = torch.load(best_path, map_location=device)
        model.load_state_dict(ckpt["model_state_dict"])

    return model, pd.DataFrame(history), best_path

### Original code cell 29

In [ ]:
def _argmax_xy_value(hm):
    """
    Return (x, y, value) of the maximum pixel in a 2D heatmap.
    Works with numpy arrays or torch tensors.
    """
    if torch.is_tensor(hm):
        hm_np = hm.detach().float().cpu().numpy()
    else:
        hm_np = np.asarray(hm)
    if hm_np.ndim != 2:
        raise ValueError(f'_argmax_xy_value expects 2D heatmap, got shape {hm_np.shape}')
    idx = int(np.argmax(hm_np))
    y, x = np.unravel_index(idx, hm_np.shape)
    return (float(x), float(y), float(hm_np[y, x]))

def _argmax_xy_conf(hm):
    """
    Return (x, y, confidence), where confidence is max value after safe conversion.
    Alias-style helper used by some decoding/evaluation cells.
    """
    return _argmax_xy_value(hm)

def _value_at_xy(hm, x, y):
    """
    Return heatmap value at rounded/clipped x,y coordinate.
    """
    if torch.is_tensor(hm):
        hm_np = hm.detach().float().cpu().numpy()
    else:
        hm_np = np.asarray(hm)
    if hm_np.ndim != 2:
        raise ValueError(f'_value_at_xy expects 2D heatmap, got shape {hm_np.shape}')
    h, w = hm_np.shape
    xi = int(np.clip(round(float(x)), 0, w - 1))
    yi = int(np.clip(round(float(y)), 0, h - 1))
    return float(hm_np[yi, xi])

def _norm01(a, eps=1e-8):
    """
    Normalize array/tensor to [0, 1].
    """
    if torch.is_tensor(a):
        x = a.detach().float().cpu().numpy()
    else:
        x = np.asarray(a, dtype=np.float32)
    mn = float(np.nanmin(x))
    mx = float(np.nanmax(x))
    if not np.isfinite(mn) or not np.isfinite(mx) or mx - mn < eps:
        return np.zeros_like(x, dtype=np.float32)
    return ((x - mn) / (mx - mn + eps)).astype(np.float32)

def _masked_heatmap(hm, mask, fill_value):
    """
    Apply boolean mask to a heatmap. Pixels outside mask get fill_value.
    If mask is None, returns hm as numpy float32.
    """
    if torch.is_tensor(hm):
        hm_np = hm.detach().float().cpu().numpy()
    else:
        hm_np = np.asarray(hm, dtype=np.float32)
    if mask is None:
        return hm_np.astype(np.float32)
    if torch.is_tensor(mask):
        mask_np = mask.detach().cpu().numpy().astype(bool)
    else:
        mask_np = np.asarray(mask).astype(bool)
    if mask_np.shape != hm_np.shape:
        raise ValueError(f'Mask shape {mask_np.shape} does not match heatmap shape {hm_np.shape}')
    out = hm_np.astype(np.float32).copy()
    out[~mask_np] = fill_value
    return out

def _relax_prior_x(prior_map, relax_px):
    """
    Relax a prior map horizontally by max-pooling over x direction.
    Useful if x-position differs between RSNA and external data.
    """
    if torch.is_tensor(prior_map):
        p = prior_map.detach().float().cpu().numpy()
    else:
        p = np.asarray(prior_map, dtype=np.float32)
    if p.ndim != 2:
        raise ValueError(f'_relax_prior_x expects 2D prior map, got shape {p.shape}')
    if relax_px <= 0:
        return p.astype(np.float32)
    h, w = p.shape
    out = np.zeros_like(p, dtype=np.float32)
    for dx in range(-int(relax_px), int(relax_px) + 1):
        if dx < 0:
            out[:, :dx] = np.maximum(out[:, :dx], p[:, -dx:])
        elif dx > 0:
            out[:, dx:] = np.maximum(out[:, dx:], p[:, :-dx])
        else:
            out = np.maximum(out, p)
    return out.astype(np.float32)

def _fuse_heatmap_with_scores(hm, score_map, alpha):
    """
    Fuse predicted heatmap with an auxiliary score/prior map.
    alpha controls score/prior strength.
    """
    hm01 = _norm01(hm)
    if score_map is None:
        return hm01
    score01 = _norm01(score_map)
    if score01.shape != hm01.shape:
        raise ValueError(f'Score map shape {score01.shape} does not match heatmap shape {hm01.shape}')
    fused = hm01 * (1.0 - float(alpha) + float(alpha) * score01)
    return _norm01(fused)

def _fuse_raw_with_prior(raw_hm, prior_map, prior_weight, prior_epsilon):
    """
    Fuse raw predicted heatmap with coordinate prior map.

    Formula:
        fused = normalized(raw_hm) * ((1 - prior_weight) + prior_weight * normalized(prior + epsilon))
    """
    raw01 = _norm01(raw_hm)
    if prior_map is None:
        return raw01
    prior01 = _norm01(np.asarray(prior_map, dtype=np.float32) + float(prior_epsilon))
    if prior01.shape != raw01.shape:
        raise ValueError(f'Prior map shape {prior01.shape} does not match heatmap shape {raw01.shape}')
    w = float(prior_weight)
    fused = raw01 * (1.0 - w + w * prior01)
    return _norm01(fused)


### Original code cell 30

In [ ]:
@torch.no_grad()
def generate_raw_heatmaps_from_loader(model, loader, device, apply_sigmoid):
    """
    Generate raw prediction heatmaps for a full dataset loader.

    Expected batch keys from your notebook:
      batch["image"]      -> [B, C, H, W]
      batch["level_idx"]  -> [B]
      batch["target"]     -> [B, 1, H, W] optional
      batch["point_xy"]   -> [B, 2] optional

    Model call:
      model(batch["image"], batch["level_idx"])

    Returns a dict with numpy arrays.
    """
    model.eval()
    all_heatmaps = []
    all_level_idx = []
    all_point_xy = []
    all_targets = []
    for batch in tqdm(loader, desc='Generating raw heatmaps'):
        images = batch['image'].float().to(device)
        level_idx = batch['level_idx'].long().to(device)
        logits = model(images, level_idx)
        if apply_sigmoid:
            preds = torch.sigmoid(logits)
        else:
            preds = logits
        preds_np = preds.detach().cpu().numpy()
        if preds_np.ndim == 4 and preds_np.shape[1] == 1:
            preds_np = preds_np[:, 0]
        all_heatmaps.append(preds_np)
        all_level_idx.append(level_idx.detach().cpu().numpy())
        if 'point_xy' in batch:
            all_point_xy.append(batch['point_xy'].detach().cpu().numpy())
        if 'target' in batch:
            target_np = batch['target'].detach().cpu().numpy()
            if target_np.ndim == 4 and target_np.shape[1] == 1:
                target_np = target_np[:, 0]
            all_targets.append(target_np)
    out = {'heatmaps': np.concatenate(all_heatmaps, axis=0), 'level_idx': np.concatenate(all_level_idx, axis=0)}
    if len(all_point_xy) > 0:
        out['point_xy'] = np.concatenate(all_point_xy, axis=0)
    if len(all_targets) > 0:
        out['targets'] = np.concatenate(all_targets, axis=0)
    return out


### Original code cell 34

In [ ]:
def tensor_to_numpy_image(image, channel):
    """
    Convert an image tensor/array to a 2D numpy image for plotting.

    Supports:
      - torch tensor [C, H, W]
      - torch tensor [H, W]
      - numpy array [C, H, W]
      - numpy array [H, W]
      - numpy array [H, W, C]

    If channel is None and the image has channels, the middle channel is used.
    """
    if torch.is_tensor(image):
        arr = image.detach().float().cpu().numpy()
    else:
        arr = np.asarray(image)
    if arr.ndim == 2:
        img = arr
    elif arr.ndim == 3:
        if arr.shape[0] <= 8 and arr.shape[1] > 8 and (arr.shape[2] > 8):
            if channel is None:
                channel = arr.shape[0] // 2
            channel = int(np.clip(channel, 0, arr.shape[0] - 1))
            img = arr[channel]
        elif arr.shape[-1] <= 8 and arr.shape[0] > 8 and (arr.shape[1] > 8):
            if channel is None:
                channel = arr.shape[-1] // 2
            channel = int(np.clip(channel, 0, arr.shape[-1] - 1))
            img = arr[..., channel]
        else:
            raise ValueError(f'Could not infer channel layout from shape {arr.shape}')
    else:
        raise ValueError(f'Expected 2D or 3D image, got shape {arr.shape}')
    return _norm01(img)


### Original code cell 36

In [ ]:
def _norm01(x, eps=1e-8):
    x = np.asarray(x, dtype=np.float32)
    mn = np.nanmin(x)
    mx = np.nanmax(x)
    if mx - mn < eps:
        return np.zeros_like(x, dtype=np.float32)
    return (x - mn) / (mx - mn + eps)

def _argmax_xy_conf(hmap):
    hmap = np.asarray(hmap)
    y, x = np.unravel_index(np.argmax(hmap), hmap.shape)
    return (float(x), float(y), float(hmap[y, x]))

def plot_raw_prior_final(idx, dataset, raw_preds, prior_maps, lam, eps, fusion_mode, alpha):
    """
    3 panels:
      1) image + raw predicted heatmap
      2) image + full prior heatmap for the same level
      3) image + final fused heatmap
    """
    sample = dataset[idx]
    image = sample['image']
    image_2d = tensor_to_numpy_image(image, channel=image.shape[0] // 2)
    raw_hmap = raw_preds['heatmaps'][idx]
    level_idx = int(raw_preds['level_idx'][idx])
    level_name = LEVELS[level_idx]
    prior_hmap = prior_maps[level_idx]
    raw_n = _norm01(raw_hmap)
    prior_n = _norm01(prior_hmap)
    if raw_n.shape != prior_n.shape:
        raise ValueError(f'Raw heatmap shape {raw_n.shape} and prior shape {prior_n.shape} do not match. Regenerate PRIOR_MAPS with img_size equal to the model heatmap size.')
    if fusion_mode == 'log':
        fused = np.log(raw_n + eps) + lam * np.log(prior_n + eps)
        fused_n = _norm01(fused)
    elif fusion_mode == 'mul':
        fused = raw_n * np.power(prior_n, lam)
        fused_n = _norm01(fused)
    else:
        raise ValueError("fusion_mode must be 'log' or 'mul'")
    tx = ty = None
    if 'point_xy' in sample:
        point_xy = sample['point_xy']
        if hasattr(point_xy, 'detach'):
            point_xy = point_xy.detach().cpu().numpy()
        tx, ty = (float(point_xy[0]), float(point_xy[1]))
    raw_x, raw_y, raw_conf = _argmax_xy_conf(raw_n)
    prior_x, prior_y, prior_conf = _argmax_xy_conf(prior_n)
    fused_x, fused_y, fused_conf = _argmax_xy_conf(fused_n)
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(image_2d, cmap='gray')
    axes[0].imshow(raw_n, cmap='hot', alpha=alpha)
    axes[0].scatter(raw_x, raw_y, c='cyan', s=70, marker='x', label='raw argmax')
    if tx is not None:
        axes[0].scatter(tx, ty, c='lime', s=60, label='GT')
    axes[0].set_title(f'Raw model prediction\nidx={idx}, level={level_name}')
    axes[0].axis('off')
    axes[0].legend(loc='lower right')
    axes[1].imshow(image_2d, cmap='gray')
    axes[1].imshow(prior_n, cmap='Blues', alpha=alpha)
    axes[1].scatter(prior_x, prior_y, c='cyan', s=70, marker='x', label='prior peak')
    if tx is not None:
        axes[1].scatter(tx, ty, c='lime', s=60, label='GT')
    axes[1].set_title(f'Full prior heatmap\n{level_name}')
    axes[1].axis('off')
    axes[1].legend(loc='lower right')
    axes[2].imshow(image_2d, cmap='gray')
    axes[2].imshow(fused_n, cmap='magma', alpha=alpha)
    axes[2].scatter(fused_x, fused_y, c='yellow', s=70, marker='x', label='final argmax')
    if tx is not None:
        axes[2].scatter(tx, ty, c='lime', s=60, label='GT')
    axes[2].set_title(f'Final fused heatmap\nmode={fusion_mode}, λ={lam}')
    axes[2].axis('off')
    axes[2].legend(loc='lower right')
    plt.tight_layout()
    plt.show()
    print(f'idx: {idx}')
    print(f'level: {level_name} (level_idx={level_idx})')
    if tx is not None:
        print(f'GT point:     ({tx:.2f}, {ty:.2f})')
    print(f'Raw argmax:   ({raw_x:.2f}, {raw_y:.2f}) conf={raw_conf:.4f}')
    print(f'Prior peak:   ({prior_x:.2f}, {prior_y:.2f}) conf={prior_conf:.4f}')
    print(f'Final argmax: ({fused_x:.2f}, {fused_y:.2f}) conf={fused_conf:.4f}')
    return {'idx': idx, 'level_idx': level_idx, 'level_name': level_name, 'image_2d': image_2d, 'raw_hmap': raw_hmap, 'prior_hmap': prior_hmap, 'fused': fused, 'raw_argmax': (raw_x, raw_y), 'prior_argmax': (prior_x, prior_y), 'final_argmax': (fused_x, fused_y)}


### Original code cell 37

In [ ]:
def _norm01(x, eps):
    x = np.asarray(x, dtype=np.float32)
    mn = np.nanmin(x)
    mx = np.nanmax(x)
    if mx - mn < eps:
        return np.zeros_like(x, dtype=np.float32)
    return (x - mn) / (mx - mn + eps)

def _argmax_xy_conf(hmap):
    hmap = np.asarray(hmap)
    y, x = np.unravel_index(np.argmax(hmap), hmap.shape)
    return (float(x), float(y), float(hmap[y, x]))

def _masked_heatmap(hmap, threshold):
    """
    Hide low heatmap values so the whole image is not tinted.
    threshold is applied after normalization to [0, 1].
    """
    h = _norm01(hmap)
    return np.ma.masked_where(h < threshold, h)

def _relax_prior_x(prior_hmap, sigma_x, sigma_y, eps):
    """
    Make prior less strict along x-axis while preserving y-axis localization.

    sigma_x: horizontal smoothing strength in pixels.
             Larger value = prior allows more left/right shift.
    sigma_y: vertical smoothing strength in pixels.
             Keep this near 0 if you want to preserve level-specific y location.
    """
    prior = _norm01(prior_hmap, eps=eps)
    relaxed = gaussian_filter(prior, sigma=(sigma_y, sigma_x), mode='constant')
    relaxed = _norm01(relaxed, eps=eps)
    relaxed = np.clip(relaxed, eps, 1.0)
    return relaxed

def _fuse_raw_with_prior(raw_hmap, prior_hmap, lam, eps, fusion_mode, prior_x_relax_sigma, prior_y_relax_sigma):
    """
    Fuse raw prediction heatmap with x-relaxed prior map.

    The prior is first horizontally relaxed, then fused.
    """
    raw_n = _norm01(raw_hmap, eps=eps)
    prior_relaxed = _relax_prior_x(prior_hmap, sigma_x=prior_x_relax_sigma, sigma_y=prior_y_relax_sigma, eps=eps)
    if raw_n.shape != prior_relaxed.shape:
        raise ValueError(f'Raw heatmap shape {raw_n.shape} and prior shape {prior_relaxed.shape} do not match.')
    if fusion_mode == 'log':
        fused = np.log(raw_n + eps) + lam * np.log(prior_relaxed + eps)
        fused_n = _norm01(fused, eps=eps)
    elif fusion_mode == 'mul':
        fused = raw_n * np.power(prior_relaxed, lam)
        fused_n = _norm01(fused, eps=eps)
    else:
        raise ValueError("fusion_mode must be 'log' or 'mul'")
    return (fused, fused_n, prior_relaxed)

def plot_raw_prior_final_masked(idx, dataset, raw_preds, prior_maps, lam, eps, fusion_mode, alpha, raw_threshold, prior_threshold, fused_threshold, prior_x_relax_sigma, prior_y_relax_sigma):
    sample = dataset[idx]
    image = sample['image']
    image_2d = tensor_to_numpy_image(image, channel=image.shape[0] // 2)
    raw_hmap = raw_preds['heatmaps'][idx]
    level_idx = int(raw_preds['level_idx'][idx])
    level_name = LEVELS[level_idx]
    prior_hmap = prior_maps[level_idx]
    raw_n = _norm01(raw_hmap, eps=eps)
    prior_n = _norm01(prior_hmap, eps=eps)
    fused, fused_n, prior_relaxed = _fuse_raw_with_prior(raw_hmap=raw_hmap, prior_hmap=prior_hmap, lam=lam, eps=eps, fusion_mode=fusion_mode, prior_x_relax_sigma=prior_x_relax_sigma, prior_y_relax_sigma=prior_y_relax_sigma)
    tx = ty = None
    if 'point_xy' in sample:
        point_xy = sample['point_xy']
        if hasattr(point_xy, 'detach'):
            point_xy = point_xy.detach().cpu().numpy()
        tx, ty = (float(point_xy[0]), float(point_xy[1]))
    raw_x, raw_y, raw_conf = _argmax_xy_conf(raw_n)
    prior_x, prior_y, prior_conf = _argmax_xy_conf(prior_relaxed)
    fused_x, fused_y, fused_conf = _argmax_xy_conf(fused_n)
    raw_masked = _masked_heatmap(raw_n, threshold=raw_threshold)
    prior_masked = _masked_heatmap(prior_relaxed, threshold=prior_threshold)
    fused_masked = _masked_heatmap(fused_n, threshold=fused_threshold)
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(image_2d, cmap='gray')
    axes[0].imshow(raw_masked, cmap='hot', alpha=alpha, vmin=raw_threshold, vmax=1.0)
    axes[0].scatter(raw_x, raw_y, c='cyan', s=70, marker='x', label='raw argmax')
    if tx is not None:
        axes[0].scatter(tx, ty, c='lime', s=60, label='GT')
    axes[0].set_title(f'Raw prediction\nidx={idx}, level={level_name}')
    axes[0].axis('off')
    axes[0].legend(loc='lower right')
    axes[1].imshow(image_2d, cmap='gray')
    axes[1].imshow(prior_masked, cmap='Blues', alpha=alpha, vmin=prior_threshold, vmax=1.0)
    axes[1].scatter(prior_x, prior_y, c='cyan', s=70, marker='x', label='relaxed prior peak')
    if tx is not None:
        axes[1].scatter(tx, ty, c='lime', s=60, label='GT')
    axes[1].set_title(f'X-relaxed prior heatmap\n{level_name}, σx={prior_x_relax_sigma}, σy={prior_y_relax_sigma}')
    axes[1].axis('off')
    axes[1].legend(loc='lower right')
    axes[2].imshow(image_2d, cmap='gray')
    axes[2].imshow(fused_masked, cmap='magma', alpha=alpha, vmin=fused_threshold, vmax=1.0)
    axes[2].scatter(fused_x, fused_y, c='yellow', s=70, marker='x', label='final argmax')
    if tx is not None:
        axes[2].scatter(tx, ty, c='lime', s=60, label='GT')
    axes[2].set_title(f'Final fused heatmap\nmode={fusion_mode}, λ={lam}, σx={prior_x_relax_sigma}')
    axes[2].axis('off')
    axes[2].legend(loc='lower right')
    plt.tight_layout()
    plt.show()
    print(f'idx: {idx}')
    print(f'level: {level_name} (level_idx={level_idx})')
    if tx is not None:
        print(f'GT point:     ({tx:.2f}, {ty:.2f})')
    print(f'Raw argmax:   ({raw_x:.2f}, {raw_y:.2f}) conf={raw_conf:.4f}')
    print(f'Prior peak:   ({prior_x:.2f}, {prior_y:.2f}) conf={prior_conf:.4f}')
    print(f'Final argmax: ({fused_x:.2f}, {fused_y:.2f}) conf={fused_conf:.4f}')
    return {'raw_hmap': raw_hmap, 'prior_hmap_original': prior_hmap, 'prior_hmap_relaxed': prior_relaxed, 'fused': fused, 'fused_n': fused_n, 'raw_argmax': (raw_x, raw_y), 'prior_argmax': (prior_x, prior_y), 'final_argmax': (fused_x, fused_y)}


### Original code cell 39

In [ ]:
def _norm01(x, eps=1e-8):
    x = np.asarray(x, dtype=np.float32)
    mn = np.nanmin(x)
    mx = np.nanmax(x)
    if mx - mn < eps:
        return np.zeros_like(x, dtype=np.float32)
    return (x - mn) / (mx - mn + eps)

def _argmax_xy_value(hmap):
    hmap = np.asarray(hmap)
    y, x = np.unravel_index(np.argmax(hmap), hmap.shape)
    return (float(x), float(y), float(hmap[y, x]))

def _value_at_xy(hmap, x, y):
    hmap = np.asarray(hmap)
    xi = int(round(x))
    yi = int(round(y))
    yi = np.clip(yi, 0, hmap.shape[0] - 1)
    xi = np.clip(xi, 0, hmap.shape[1] - 1)
    return float(hmap[yi, xi])

def _relax_prior_x(prior_hmap, sigma_x, sigma_y, eps):
    prior = _norm01(prior_hmap, eps=eps)
    relaxed = gaussian_filter(prior, sigma=(sigma_y, sigma_x), mode='constant')
    relaxed = _norm01(relaxed, eps=eps)
    relaxed = np.clip(relaxed, eps, 1.0)
    return relaxed

def _fuse_heatmap_with_scores(raw_hmap, prior_hmap, lam, mode, eps, prior_x_relax_sigma, prior_y_relax_sigma):
    """
    Returns:
      fused_score: unnormalized score used for ranking
      fused_n: normalized score for plotting only
      raw_n: normalized raw heatmap
      prior_relaxed: normalized x-relaxed prior
    """
    raw_n = _norm01(raw_hmap, eps=1e-8)
    prior_relaxed = _relax_prior_x(prior_hmap, sigma_x=prior_x_relax_sigma, sigma_y=prior_y_relax_sigma, eps=eps)
    if raw_n.shape != prior_relaxed.shape:
        raise ValueError(f'Raw heatmap shape {raw_n.shape} and prior shape {prior_relaxed.shape} do not match.')
    if mode == 'log':
        fused_score = np.log(raw_n + eps) + lam * np.log(prior_relaxed + eps)
    elif mode == 'mul':
        fused_score = raw_n * np.power(prior_relaxed, lam)
    else:
        raise ValueError("mode must be 'log' or 'mul'")
    fused_n = _norm01(fused_score, eps=eps)
    return (fused_score, fused_n, raw_n, prior_relaxed)

def _make_peak_map(points, shape, radius):
    """
    Create a simple peak-only image from point list.
    points: list of dicts with keys x, y, conf
    """
    h, w = shape
    peak_map = np.zeros((h, w), dtype=np.float32)
    for p in points:
        x = int(round(p['x']))
        y = int(round(p['y']))
        conf = float(p['conf'])
        y0 = max(0, y - radius)
        y1 = min(h, y + radius + 1)
        x0 = max(0, x - radius)
        x1 = min(w, x + radius + 1)
        peak_map[y0:y1, x0:x1] = np.maximum(peak_map[y0:y1, x0:x1], conf)
    return peak_map

def plot_gt_vs_fused_peaks(anchor_idx, dataset, raw_preds, prior_maps, lam, fusion_mode, eps, group_cols, prior_x_relax_sigma, prior_y_relax_sigma, tolerance_px):
    """
    LEFT  = ground-truth points
    RIGHT = fused prediction peaks using x-relaxed prior

    Adds:
      - dist_px
      - ok_5px
      - raw_conf_at_pred
      - prior_at_pred
      - fused_score_at_pred
    """
    if not hasattr(dataset, 'df'):
        raise AttributeError('dataset.df is required for grouping samples.')
    df = dataset.df.reset_index(drop=True)
    if group_cols is None:
        preferred = ['study_id', 'series_id', 'instance_number', 'condition']
        group_cols = [c for c in preferred if c in df.columns]
        if len(group_cols) == 0:
            raise ValueError('Could not infer grouping columns. Please pass group_cols manually.')
    row = df.iloc[anchor_idx]
    mask = pd.Series(True, index=df.index)
    for c in group_cols:
        mask &= df[c] == row[c]
    group_indices = df.index[mask].tolist()
    if len(group_indices) == 0:
        raise ValueError('No rows found for selected anchor_idx group.')
    if 'level_idx' in df.columns:
        group_indices = sorted(group_indices, key=lambda i: int(df.loc[i, 'level_idx']))
    else:
        level_to_idx = {lvl: i for i, lvl in enumerate(LEVELS)}
        group_indices = sorted(group_indices, key=lambda i: level_to_idx[str(df.loc[i, 'level'])])
    sample0 = dataset[group_indices[0]]
    image = sample0['image']
    image_2d = tensor_to_numpy_image(image, channel=image.shape[0] // 2)
    rows = []
    for ds_idx in group_indices:
        sample_i = dataset[ds_idx]
        gt_x = gt_y = np.nan
        if 'point_xy' in sample_i:
            point_xy = sample_i['point_xy']
            if hasattr(point_xy, 'detach'):
                point_xy = point_xy.detach().cpu().numpy()
            gt_x, gt_y = (float(point_xy[0]), float(point_xy[1]))
        raw_hmap = raw_preds['heatmaps'][ds_idx]
        level_idx = int(raw_preds['level_idx'][ds_idx])
        level_name = LEVELS[level_idx]
        prior_hmap = prior_maps[level_idx]
        fused_score, fused_n, raw_n, prior_relaxed = _fuse_heatmap_with_scores(raw_hmap=raw_hmap, prior_hmap=prior_hmap, lam=lam, mode=fusion_mode, eps=eps, prior_x_relax_sigma=prior_x_relax_sigma, prior_y_relax_sigma=prior_y_relax_sigma)
        pred_x, pred_y, fused_score_at_pred = _argmax_xy_value(fused_score)
        raw_conf_at_pred = _value_at_xy(raw_n, pred_x, pred_y)
        prior_at_pred = _value_at_xy(prior_relaxed, pred_x, pred_y)
        fused_norm_at_pred = _value_at_xy(fused_n, pred_x, pred_y)
        if np.isfinite(gt_x) and np.isfinite(gt_y):
            dist_px = float(np.sqrt((pred_x - gt_x) ** 2 + (pred_y - gt_y) ** 2))
            ok_tol = dist_px <= tolerance_px
        else:
            dist_px = np.nan
            ok_tol = False
        rows.append({'dataset_idx': ds_idx, 'level_idx': level_idx, 'level': level_name, 'gt_x': gt_x, 'gt_y': gt_y, 'pred_x': pred_x, 'pred_y': pred_y, 'dist_px': dist_px, f'ok_{int(tolerance_px)}px': ok_tol, 'raw_conf_at_pred': raw_conf_at_pred, 'prior_at_pred': prior_at_pred, 'fused_score_at_pred': fused_score_at_pred, 'fused_norm_at_pred': fused_norm_at_pred})
    peak_df = pd.DataFrame(rows).sort_values('level_idx').reset_index(drop=True)
    colors = ['cyan', 'yellow', 'lime', 'orange', 'red']
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(image_2d, cmap='gray')
    for i, row_i in peak_df.iterrows():
        c = colors[i % len(colors)]
        axes[0].scatter(row_i['gt_x'], row_i['gt_y'], c=c, s=80, marker='o')
        axes[0].text(row_i['gt_x'] + 3, row_i['gt_y'] - 3, row_i['level'], color=c, fontsize=10, weight='bold')
    axes[0].set_title('Ground truth')
    axes[0].axis('off')
    axes[1].imshow(image_2d, cmap='gray')
    for i, row_i in peak_df.iterrows():
        c = colors[i % len(colors)]
        marker_color = c if row_i[f'ok_{int(tolerance_px)}px'] else 'red'
        axes[1].scatter(row_i['pred_x'], row_i['pred_y'], c=marker_color, s=90, marker='x')
        axes[1].text(row_i['pred_x'] + 3, row_i['pred_y'] - 3, f"{row_i['level']} ({row_i['dist_px']:.1f}px)", color=marker_color, fontsize=9, weight='bold')
    axes[1].set_title(f'Fused prediction peaks\nmode={fusion_mode}, λ={lam}, σx={prior_x_relax_sigma}, tol={tolerance_px}px')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()
    print('Grouping columns used:', group_cols)
    print('Anchor idx:', anchor_idx)
    print('Matched rows:', len(group_indices))
    print(f'\nWithin {tolerance_px}px:')
    print(f"{peak_df[f'ok_{int(tolerance_px)}px'].sum()} / {len(peak_df)}")
    return peak_df


### Original code cell 40

In [ ]:

def get_first_index_per_series(dataset):
    df = dataset.df.reset_index(drop=True)

    if "series_id" not in df.columns:
        raise ValueError("dataset.df does not contain 'series_id'.")

    first_indices = (
        df.reset_index()
          .groupby("series_id")["index"]
          .first()
          .tolist()
    )

    return first_indices


### Original code cell 45

In [ ]:

def get_inference_anchor_indices(dataset, group_cols=None):
    """
    One anchor row per unique image/series group.

    Default grouping follows the current notebook structure:
      study_id + series_id + instance_number + condition

    Each anchor image will be used to predict all 5 levels.
    """
    df = dataset.df.reset_index(drop=True)
    if group_cols is None:
        preferred = ['study_id', 'series_id', 'instance_number', 'condition']
        group_cols = [c for c in preferred if c in df.columns]
    if len(group_cols) == 0:
        raise ValueError('No grouping columns found. Please pass group_cols manually.')
    anchor_df = df.reset_index().groupby(group_cols, dropna=False)['index'].first().reset_index().rename(columns={'index': 'anchor_idx'})
    return (anchor_df, group_cols)


### Original code cell 47

In [ ]:
@torch.no_grad()
def predict_all_5_levels_for_anchor(anchor_idx, dataset, model, device, prior_maps, lam, fusion_mode, prior_x_relax_sigma, prior_y_relax_sigma, eps):
    model.eval()
    df = dataset.df.reset_index(drop=True)
    row_meta = df.loc[anchor_idx].to_dict()
    sample = dataset[anchor_idx]
    image = sample['image']
    image_batch = image.unsqueeze(0).float().to(device)
    image_batch_5 = image_batch.repeat(len(LEVELS), 1, 1, 1)
    level_idx_batch = torch.arange(len(LEVELS), dtype=torch.long, device=device)
    logits = model(image_batch_5, level_idx_batch)
    raw_preds = torch.sigmoid(logits).detach().cpu().numpy()
    if raw_preds.ndim == 4 and raw_preds.shape[1] == 1:
        raw_preds = raw_preds[:, 0]
    rows = []
    raw_heatmaps = {}
    for level_idx, level_name in enumerate(LEVELS):
        raw_hmap = raw_preds[level_idx]
        prior_hmap = prior_maps[level_idx]
        fused_score, fused_n, raw_n, prior_relaxed = _fuse_heatmap_with_scores(raw_hmap=raw_hmap, prior_hmap=prior_hmap, lam=lam, mode=fusion_mode, eps=eps, prior_x_relax_sigma=prior_x_relax_sigma, prior_y_relax_sigma=prior_y_relax_sigma)
        raw_heatmaps[int(anchor_idx), int(level_idx)] = raw_n
        pred_x, pred_y, fused_score_at_pred = _argmax_xy_value(fused_score)
        raw_conf_at_pred = _value_at_xy(raw_n, pred_x, pred_y)
        prior_at_pred = _value_at_xy(prior_relaxed, pred_x, pred_y)
        fused_norm_at_pred = _value_at_xy(fused_n, pred_x, pred_y)
        out = {'anchor_idx': anchor_idx, 'level_idx': level_idx, 'level': level_name, 'pred_x': pred_x, 'pred_y': pred_y, 'raw_conf_at_pred': raw_conf_at_pred, 'prior_at_pred': prior_at_pred, 'fused_score_at_pred': fused_score_at_pred, 'fused_norm_at_pred': fused_norm_at_pred}
        for c in ['study_id', 'series_id', 'instance_number', 'condition', 'series_description', 'img_path']:
            if c in row_meta:
                out[c] = row_meta[c]
        rows.append(out)
    return (rows, raw_heatmaps)


### Original code cell 48

In [ ]:
def predict_all_5_levels_for_dataset(dataset, model, device, prior_maps, anchor_df, lam, fusion_mode, prior_x_relax_sigma, prior_y_relax_sigma, eps):
    if anchor_df is None:
        anchor_df, group_cols = get_inference_anchor_indices(dataset)
    else:
        group_cols = None
    all_rows = []
    raw_heatmaps = {}
    for anchor_idx in tqdm(anchor_df['anchor_idx'].tolist(), desc='Predicting all 5 levels'):
        rows, anchor_raw_heatmaps = predict_all_5_levels_for_anchor(anchor_idx=anchor_idx, dataset=dataset, model=model, device=device, prior_maps=prior_maps, lam=lam, fusion_mode=fusion_mode, prior_x_relax_sigma=prior_x_relax_sigma, prior_y_relax_sigma=prior_y_relax_sigma, eps=eps)
        all_rows.extend(rows)
        raw_heatmaps.update(anchor_raw_heatmaps)
    pred_df = pd.DataFrame(all_rows)
    sort_cols = [c for c in ['study_id', 'series_id', 'instance_number', 'condition', 'level_idx'] if c in pred_df.columns]
    pred_df = pred_df.sort_values(sort_cols).reset_index(drop=True)
    return (pred_df, raw_heatmaps)


### Original code cell 50

In [ ]:
def plot_all5_predictions_for_anchor(anchor_idx, pred_df, dataset, figsize=(5,5)):
    sample = dataset[anchor_idx]
    image = sample['image']
    image_2d = tensor_to_numpy_image(image, channel=image.shape[0] // 2)
    row = dataset.df.reset_index(drop=True).loc[anchor_idx]
    mask = pred_df['anchor_idx'].eq(anchor_idx)
    one = pred_df[mask].sort_values('level_idx')
    colors = ['cyan', 'yellow', 'lime', 'orange', 'red']
    plt.figure(figsize=figsize)
    plt.imshow(image_2d, cmap='gray')
    for i, (_, r) in enumerate(one.iterrows()):
        c = colors[i % len(colors)]
        plt.scatter(r['pred_x'], r['pred_y'], c=c, s=90, marker='x')
        plt.text(r['pred_x'] + 3, r['pred_y'] - 3, r['level'], color=c, fontsize=10, weight='bold')
    title_parts = []
    for c in ['study_id', 'series_id', 'instance_number', 'condition']:
        if c in row:
            title_parts.append(f'{c}={row[c]}')
    plt.title('All 5 predicted levels\n' + ' | '.join(title_parts))
    plt.axis('off')
    plt.show()


### Original code cell 54

In [ ]:
def build_training_euclidean_spacing_constraints(train_df, levels, group_cols, condition_filter, min_valid_dist):
    """
    Build pair-specific relative Euclidean spacing constraints from training GT.

    For each complete 5-level group:
      dx_i = x_{i+1} - x_i
      dy_i = y_{i+1} - y_i
      dist_i = sqrt(dx_i^2 + dy_i^2)
      ref = median(dist_i)
      ratio_i = dist_i / ref

    Bounds:
      lower = min(ratio_i) - 0.5 * IQR
      upper = max(ratio_i) + 0.5 * IQR

    Uses normalized coordinates:
      x_norm, y_norm
    """
    df = train_df.copy()
    if condition_filter is not None and 'condition' in df.columns:
        df = df[df['condition'] == condition_filter].copy()
    if group_cols is None:
        preferred = ['study_id', 'series_id', 'instance_number', 'condition']
        group_cols = [c for c in preferred if c in df.columns]
    if len(group_cols) == 0:
        raise ValueError('No grouping columns found. Please pass group_cols manually.')
    required = group_cols + ['level', 'x_norm', 'y_norm']
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f'Missing required columns: {missing}')
    level_to_idx = {lvl: i for i, lvl in enumerate(levels)}
    df['level_idx_tmp'] = df['level'].map(level_to_idx)
    ratio_rows = []
    for _, g in df.groupby(group_cols, dropna=False):
        g = g.dropna(subset=['level_idx_tmp', 'x_norm', 'y_norm']).copy()
        g = g.sort_values('level_idx_tmp')
        g = g.drop_duplicates('level_idx_tmp', keep='first')
        present = set(g['level_idx_tmp'].astype(int).tolist())
        if present != set(range(len(levels))):
            continue
        x_by_level = {int(row['level_idx_tmp']): float(row['x_norm']) for _, row in g.iterrows()}
        y_by_level = {int(row['level_idx_tmp']): float(row['y_norm']) for _, row in g.iterrows()}
        xs = np.array([x_by_level[i] for i in range(len(levels))], dtype=np.float32)
        ys = np.array([y_by_level[i] for i in range(len(levels))], dtype=np.float32)
        dx = np.diff(xs)
        dy = np.diff(ys)
        dist = np.sqrt(dx ** 2 + dy ** 2)
        if np.any(dy <= 0):
            continue
        if np.any(dist <= min_valid_dist):
            continue
        ref = float(np.median(dist))
        if ref <= min_valid_dist:
            continue
        ratios = dist / ref
        for i, ratio in enumerate(ratios):
            ratio_rows.append({'from_level_idx': i, 'to_level_idx': i + 1, 'from_level': levels[i], 'to_level': levels[i + 1], 'pair': f'{levels[i]}→{levels[i + 1]}', 'dx_rel': float(dx[i]), 'dy_rel': float(dy[i]), 'dist_rel': float(dist[i]), 'ref_dist_rel': ref, 'ratio_to_median_dist': float(ratio)})
    ratio_df = pd.DataFrame(ratio_rows)
    if len(ratio_df) == 0:
        raise ValueError('No complete 5-level training groups found for Euclidean ratio constraints.')
    constraint_rows = []
    for pair, g in ratio_df.groupby('pair'):
        ratios = g['ratio_to_median_dist'].dropna()
        q1 = float(ratios.quantile(0.25))
        q3 = float(ratios.quantile(0.75))
        iqr = q3 - q1
        ratio_min = float(ratios.min())
        ratio_max = float(ratios.max())
        ratio_low = max(0.0, ratio_min - 0.5 * iqr)
        ratio_high = ratio_max + 0.5 * iqr
        constraint_rows.append({'pair': pair, 'from_level': g['from_level'].iloc[0], 'to_level': g['to_level'].iloc[0], 'from_level_idx': int(g['from_level_idx'].iloc[0]), 'to_level_idx': int(g['to_level_idx'].iloc[0]), 'n': len(g), 'ratio_low': float(ratio_low), 'ratio_high': float(ratio_high), 'ratio_min': ratio_min, 'ratio_max': ratio_max, 'ratio_q1': q1, 'ratio_q3': q3, 'ratio_iqr': float(iqr), 'ratio_median': float(ratios.median()), 'ratio_mean': float(ratios.mean()), 'dist_rel_median': float(g['dist_rel'].median()), 'dist_rel_mean': float(g['dist_rel'].mean())})
    constraints_df = pd.DataFrame(constraint_rows).sort_values('from_level_idx').reset_index(drop=True)
    return (ratio_df, constraints_df)


### Original code cell 56

In [ ]:
def plot_training_euclidean_ratio_boxplots(
    ratio_df,
    constraints_df,
    levels=LEVELS,
):
    pair_order = [f"{levels[i]}→{levels[i+1]}" for i in range(len(levels) - 1)]

    data = [
        ratio_df.loc[ratio_df["pair"] == pair, "ratio_to_median_dist"].dropna().values
        for pair in pair_order
    ]

    plt.figure(figsize=(10, 5))
    plt.boxplot(data, tick_labels=pair_order, showfliers=True)

    for i, pair in enumerate(pair_order, start=1):
        row = constraints_df[constraints_df["pair"] == pair].iloc[0]

        plt.plot([i - 0.25, i + 0.25], [row["ratio_low"], row["ratio_low"]], color="green", linewidth=2)
        plt.plot([i - 0.25, i + 0.25], [row["ratio_high"], row["ratio_high"]], color="red", linewidth=2)

        plt.plot([i - 0.15, i + 0.15], [row["ratio_min"], row["ratio_min"]], color="green", linewidth=1, linestyle="--")
        plt.plot([i - 0.15, i + 0.15], [row["ratio_max"], row["ratio_max"]], color="red", linewidth=1, linestyle="--")

    plt.axhline(1.0, linestyle="--", linewidth=1)
    plt.ylabel("Euclidean dist_i / median(dist within image)")
    plt.title("Training adjacent Euclidean-spacing ratios\nBounds = min - 0.5*IQR / max + 0.5*IQR")
    plt.grid(True, axis="y", alpha=0.3)
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()


### Original code cell 57

In [ ]:
def diagnose_all5_euclidean_relative_spacing(pred_df, ratio_constraints, img_size, group_cols, levels, level_col, level_idx_col, x_col, y_col, order_margin_px, min_ref_dist_px):
    """
    Diagnose prediction groups using within-image relative Euclidean spacing ratios.

    For each predicted all-5 group:
      dx_i = x_{i+1} - x_i
      dy_i = y_{i+1} - y_i
      dist_i = sqrt(dx_i^2 + dy_i^2)
      ref = median(dist_i)
      ratio_i = dist_i / ref

    Pair is flagged if:
      ratio_i < ratio_low(pair)
      or ratio_i > ratio_high(pair)

    Also checks basic y-order violation with order_margin_px.
    """
    df = pred_df.copy()
    if group_cols is None:
        preferred = ['study_id', 'series_id', 'instance_number', 'condition']
        group_cols = [c for c in preferred if c in df.columns]
    if len(group_cols) == 0:
        raise ValueError('No grouping columns found. Please pass group_cols manually.')
    required = group_cols + [level_col, level_idx_col, x_col, y_col]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f'Missing required columns: {missing}')
    required_constraints = ['from_level_idx', 'ratio_low', 'ratio_high']
    missing_constraints = [c for c in required_constraints if c not in ratio_constraints.columns]
    if missing_constraints:
        raise ValueError(f'Missing required columns in ratio_constraints: {missing_constraints}')
    constraint_map = {int(row['from_level_idx']): row for _, row in ratio_constraints.iterrows()}
    expected_level_idx = list(range(len(levels)))
    group_rows = []
    pair_rows = []
    for group_key, g in df.groupby(group_cols, dropna=False):
        if not isinstance(group_key, tuple):
            group_key = (group_key,)
        group_meta = dict(zip(group_cols, group_key))
        g = g.sort_values(level_idx_col).reset_index(drop=True)
        present_level_idx = g[level_idx_col].astype(int).tolist()
        has_all5 = present_level_idx == expected_level_idx
        xs = g[x_col].astype(float).values
        ys = g[y_col].astype(float).values
        level_idxs = g[level_idx_col].astype(int).values
        level_names = g[level_col].astype(str).values
        if len(g) < 2:
            group_rows.append({**group_meta, 'n_levels': len(g), 'has_all5': has_all5, 'diagnosis_bad': True, 'order_ok': False, 'euclidean_spacing_ok': False, 'n_order_bad_pairs': np.nan, 'n_ratio_bad_pairs': np.nan, 'median_dist_px': np.nan, 'predicted_order_by_y': g.sort_values(y_col)[level_col].tolist(), 'expected_order': list(levels)})
            continue
        dx = np.diff(xs)
        dy = np.diff(ys)
        dist = np.sqrt(dx ** 2 + dy ** 2)
        median_dist_px = float(np.median(dist))
        invalid_ref = median_dist_px <= min_ref_dist_px
        order_bad_flags = []
        ratio_bad_flags = []
        for j in range(len(dist)):
            from_idx = int(level_idxs[j])
            to_idx = int(level_idxs[j + 1])
            y_current = ys[j]
            y_next = ys[j + 1]
            dx_px = float(dx[j])
            dy_px = float(dy[j])
            dist_px = float(dist[j])
            order_violation_px = max(0.0, y_current + order_margin_px - y_next)
            order_bad = order_violation_px > 0
            ratio = np.nan
            ratio_low = np.nan
            ratio_high = np.nan
            ratio_min = np.nan
            ratio_max = np.nan
            ratio_iqr = np.nan
            ratio_bad = False
            c = constraint_map.get(from_idx, None)
            if not invalid_ref and c is not None and (to_idx == from_idx + 1):
                ratio = dist_px / median_dist_px
                ratio_low = float(c['ratio_low'])
                ratio_high = float(c['ratio_high'])
                if 'ratio_min' in c:
                    ratio_min = float(c['ratio_min'])
                if 'ratio_max' in c:
                    ratio_max = float(c['ratio_max'])
                if 'ratio_iqr' in c:
                    ratio_iqr = float(c['ratio_iqr'])
                ratio_bad = ratio < ratio_low or ratio > ratio_high
            elif invalid_ref:
                ratio_bad = True
            order_bad_flags.append(order_bad)
            ratio_bad_flags.append(ratio_bad)
            pair_rows.append({**group_meta, 'from_level_idx': from_idx, 'to_level_idx': to_idx, 'from_level': level_names[j], 'to_level': level_names[j + 1], 'x_from': float(xs[j]), 'y_from': float(ys[j]), 'x_to': float(xs[j + 1]), 'y_to': float(ys[j + 1]), 'dx_px': dx_px, 'dy_px': dy_px, 'dist_px': dist_px, 'median_dist_px': median_dist_px, 'ratio_to_group_median_dist': ratio, 'ratio_low': ratio_low, 'ratio_high': ratio_high, 'ratio_min_train': ratio_min, 'ratio_max_train': ratio_max, 'ratio_iqr_train': ratio_iqr, 'ratio_bad': ratio_bad, 'order_violation_px': order_violation_px, 'order_bad': order_bad, 'any_pair_bad': order_bad or ratio_bad})
        order_ok = not any(order_bad_flags)
        euclidean_spacing_ok = not any(ratio_bad_flags)
        group_rows.append({**group_meta, 'n_levels': len(g), 'has_all5': has_all5, 'present_levels': level_names.tolist(), 'diagnosis_bad': not order_ok or not euclidean_spacing_ok, 'order_ok': order_ok, 'euclidean_spacing_ok': euclidean_spacing_ok, 'n_order_bad_pairs': int(np.sum(order_bad_flags)), 'n_ratio_bad_pairs': int(np.sum(ratio_bad_flags)), 'median_dist_px': median_dist_px, 'min_dist_px': float(np.min(dist)), 'max_dist_px': float(np.max(dist)), 'predicted_order_by_y': g.sort_values(y_col)[level_col].tolist(), 'expected_order': list(levels)})
    group_diag_df = pd.DataFrame(group_rows)
    pair_diag_df = pd.DataFrame(pair_rows)
    return (group_diag_df, pair_diag_df)


### Original code cell 60

In [ ]:
def plot_study_gt_vs_predictions(study_id, dataset, pred_df, condition_filter, series_id, instance_number, levels, figsize):
    """
    Study-level visualization:
      LEFT  = available ground truth annotations
      RIGHT = predicted all-5 points

    Optional filters:
      - condition_filter
      - series_id
      - instance_number
    """
    ds_df = dataset.df.reset_index(drop=True).copy()
    pred = pred_df.copy()
    gt_mask = ds_df['study_id'].eq(study_id)
    if condition_filter is not None and 'condition' in ds_df.columns:
        gt_mask &= ds_df['condition'].eq(condition_filter)
    if series_id is not None and 'series_id' in ds_df.columns:
        gt_mask &= ds_df['series_id'].eq(series_id)
    if instance_number is not None and 'instance_number' in ds_df.columns:
        gt_mask &= ds_df['instance_number'].eq(instance_number)
    gt_rows = ds_df[gt_mask].copy()
    if len(gt_rows) == 0:
        raise ValueError(f'No GT rows found for study_id={study_id} with selected filters.')
    anchor_idx = int(gt_rows.index[0])
    sample = dataset[anchor_idx]
    image = sample['image']
    image_2d = tensor_to_numpy_image(image, channel=image.shape[0] // 2)
    pred_mask = pred['study_id'].eq(study_id)
    if condition_filter is not None and 'condition' in pred.columns:
        pred_mask &= pred['condition'].eq(condition_filter)
    if series_id is not None and 'series_id' in pred.columns:
        pred_mask &= pred['series_id'].eq(series_id)
    if instance_number is not None and 'instance_number' in pred.columns:
        pred_mask &= pred['instance_number'].eq(instance_number)
    pred_rows = pred[pred_mask].copy()
    if len(pred_rows) == 0:
        raise ValueError(f'No prediction rows found for study_id={study_id} with selected filters.')
    bg_meta = ds_df.loc[anchor_idx]
    for c in ['series_id', 'instance_number', 'condition']:
        if c in pred_rows.columns and c in bg_meta.index:
            pred_rows = pred_rows[pred_rows[c].eq(bg_meta[c])]
    level_to_idx = {lvl: i for i, lvl in enumerate(levels)}
    if 'level_idx' in gt_rows.columns:
        gt_rows = gt_rows.sort_values('level_idx')
    else:
        gt_rows['level_idx_tmp'] = gt_rows['level'].map(level_to_idx)
        gt_rows = gt_rows.sort_values('level_idx_tmp')
    pred_rows = pred_rows.sort_values('level_idx')
    colors = ['cyan', 'yellow', 'lime', 'orange', 'red']
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    axes[0].imshow(image_2d, cmap='gray')
    for _, r in gt_rows.iterrows():
        level = str(r['level'])
        level_idx = level_to_idx.get(level, int(r['level_idx']) if 'level_idx' in r else 0)
        c = colors[level_idx % len(colors)]
        if 'x_norm' in r and 'y_norm' in r:
            gt_x = float(r['x_norm']) * (IMG_SIZE - 1)
            gt_y = float(r['y_norm']) * (IMG_SIZE - 1)
        elif 'x' in r and 'y' in r:
            gt_x = float(r['x'])
            gt_y = float(r['y'])
        else:
            continue
        axes[0].scatter(gt_x, gt_y, c=c, s=80, marker='o')
        axes[0].text(gt_x + 3, gt_y - 3, level, color=c, fontsize=10, weight='bold')
    axes[0].set_title('Ground truth')
    axes[0].axis('off')
    axes[1].imshow(image_2d, cmap='gray')
    for _, r in pred_rows.iterrows():
        level = str(r['level'])
        level_idx = int(r['level_idx'])
        c = colors[level_idx % len(colors)]
        pred_x = float(r['pred_x'])
        pred_y = float(r['pred_y'])
        axes[1].scatter(pred_x, pred_y, c=c, s=90, marker='x')
        axes[1].text(pred_x + 3, pred_y - 3, level, color=c, fontsize=10, weight='bold')
    title_bits = [f'study_id={study_id}']
    for c in ['series_id', 'instance_number', 'condition']:
        if c in bg_meta.index:
            title_bits.append(f'{c}={bg_meta[c]}')
    axes[1].set_title('Predicted all-5 points')
    axes[1].axis('off')
    plt.suptitle(' | '.join(title_bits), fontsize=13)
    plt.tight_layout()
    plt.show()
    return {'anchor_idx': anchor_idx, 'gt_rows': gt_rows, 'pred_rows': pred_rows, 'background_meta': bg_meta}


### Original code cell 69

In [ ]:
def euclid_xy(a, b):
    return float(np.linalg.norm(np.asarray(a, dtype=float) - np.asarray(b, dtype=float)))

def is_above(upper_xy, lower_xy, tol):
    return float(upper_xy[1]) < float(lower_xy[1]) - tol

def get_ratio_bounds(euclidean_ratio_constraints, from_level, to_level):
    row = euclidean_ratio_constraints[(euclidean_ratio_constraints['from_level'] == from_level) & (euclidean_ratio_constraints['to_level'] == to_level)]
    if row.empty:
        raise ValueError(f'No ratio constraint found for {from_level} → {to_level}')
    return (float(row.iloc[0]['ratio_min']), float(row.iloc[0]['ratio_max']))

def top_raw_heatmap_peaks(raw_hmap, top_k, nms_radius):
    """
    Extract top-k candidates from the already-computed raw heatmap.
    No model call. No fusion. No prior.
    """
    h = raw_hmap.astype(np.float32).copy()
    H, W = h.shape
    peaks = []
    for _ in range(top_k):
        flat_idx = int(np.argmax(h))
        conf = float(h.flat[flat_idx])
        y, x = np.unravel_index(flat_idx, h.shape)
        peaks.append({'x': float(x), 'y': float(y), 'confidence': conf})
        y0 = max(0, y - nms_radius)
        y1 = min(H, y + nms_radius + 1)
        x0 = max(0, x - nms_radius)
        x1 = min(W, x + nms_radius + 1)
        h[y0:y1, x0:x1] = -np.inf
    return peaks


### Original code cell 70

In [ ]:
def get_union_raw_heatmap_for_anchor(raw_heatmaps, anchor_idx, level_indices, mode):
    """
    Build a union raw heatmap from existing raw_heatmaps.

    raw_heatmaps[(anchor_idx, level_idx)] = 2D heatmap
    """
    anchor_idx = int(anchor_idx)
    if level_indices is None:
        level_indices = [0, 1, 2, 3, 4]
    maps = []
    for lvl_idx in level_indices:
        key = (anchor_idx, int(lvl_idx))
        if key not in raw_heatmaps:
            continue
        maps.append(raw_heatmaps[key])
    if len(maps) == 0:
        raise KeyError(f'No raw heatmaps found for anchor_idx={anchor_idx}')
    stack = np.stack(maps, axis=0)
    if mode == 'max':
        return np.max(stack, axis=0)
    if mode == 'mean':
        return np.mean(stack, axis=0)
    raise ValueError("mode must be 'max' or 'mean'")

def top_peaks_in_distance_range(hmap, lower_xy, min_allowed, max_allowed, *, top_k=CORRECTION_TOP_K, nms_radius=CORRECTION_NMS_RADIUS, min_conf):
    """
    Search top peaks only inside the distance band around lower_xy.

    Candidate must satisfy:
        min_allowed <= euclidean(candidate, lower_xy) <= max_allowed
        candidate_y < lower_y
    """
    h = hmap.astype(np.float32).copy()
    H, W = h.shape
    yy, xx = np.mgrid[0:H, 0:W]
    dx = xx - float(lower_xy[0])
    dy = yy - float(lower_xy[1])
    dist = np.sqrt(dx * dx + dy * dy)
    above_mask = yy < float(lower_xy[1]) - 3.0
    range_mask = (dist >= min_allowed) & (dist <= max_allowed)
    valid_mask = above_mask & range_mask
    h[~valid_mask] = -np.inf
    peaks = []
    for _ in range(top_k):
        flat_idx = int(np.argmax(h))
        conf = float(h.flat[flat_idx])
        if not np.isfinite(conf):
            break
        if min_conf is not None and conf < min_conf:
            break
        y, x = np.unravel_index(flat_idx, h.shape)
        cand_xy = np.array([float(x), float(y)], dtype=float)
        cand_dist = euclid_xy(cand_xy, lower_xy)
        peaks.append({'x': float(x), 'y': float(y), 'confidence': conf, 'candidate_dist': cand_dist})
        y0 = max(0, y - nms_radius)
        y1 = min(H, y + nms_radius + 1)
        x0 = max(0, x - nms_radius)
        x1 = min(W, x + nms_radius + 1)
        h[y0:y1, x0:x1] = -np.inf
    return peaks


### Original code cell 71

In [ ]:
def correct_using_union_raw_heatmaps_top3(test_all5_pred_df, group_diag_euclid_df, euclidean_ratio_constraints, raw_heatmaps,tol, *, top_k=CORRECTION_TOP_K, nms_radius=CORRECTION_NMS_RADIUS, min_candidate_score=CORRECTION_MIN_CANDIDATE_SCORE, union_mode=CORRECTION_UNION_MODE, only_diagnosis_bad):
    """
    Leakage-safe correction using existing raw heatmaps only.

    Does NOT use bad_pred_with_diag or ground truth.

    Candidate source:
        union of all 5 raw heatmaps for the same anchor.

    Candidate search:
        restricted to the allowed distance band from the trusted lower point.

    Correction start:
        determined from failed adjacent prediction pairs, not from GT-bad rows.

    Inputs required:
        test_all5_pred_df:
            anchor_idx, level_idx, level, pred_x, pred_y,
            study_id, series_id, instance_number, condition

        group_diag_euclid_df:
            study_id, series_id, instance_number, condition,
            diagnosis_bad, median_dist_px

        euclidean_ratio_constraints:
            from_level, to_level, ratio_min, ratio_max

        raw_heatmaps:
            raw_heatmaps[(anchor_idx, level_idx)] = 2D raw heatmap

    Returns:
        corrected_df, correction_report_df
    """
    corrected_df = test_all5_pred_df.copy()
    reports = []
    if only_diagnosis_bad:
        suspicious_groups = group_diag_euclid_df[group_diag_euclid_df['diagnosis_bad'] == True].copy()
    else:
        suspicious_groups = group_diag_euclid_df.copy()
    if suspicious_groups.empty:
        return (corrected_df, pd.DataFrame())
    anchor_meta_cols = ['study_id', 'series_id', 'instance_number', 'condition']
    suspicious_anchor_df = test_all5_pred_df.merge(suspicious_groups[anchor_meta_cols + ['median_dist_px']], on=anchor_meta_cols, how='inner')
    suspicious_anchor_idxs = suspicious_anchor_df['anchor_idx'].dropna().astype(int).unique()
    for anchor_idx in tqdm(suspicious_anchor_idxs, desc='Correcting from union raw heatmaps'):
        anchor_idx = int(anchor_idx)
        anchor_df = corrected_df[corrected_df['anchor_idx'] == anchor_idx].sort_values('level_idx').copy()
        if len(anchor_df) != 5:
            reports.append({'anchor_idx': anchor_idx, 'status': 'skipped_missing_all5_predictions', 'n_rows': len(anchor_df)})
            continue
        meta = anchor_df.iloc[0]
        group_row = group_diag_euclid_df[(group_diag_euclid_df['study_id'] == meta['study_id']) & (group_diag_euclid_df['series_id'] == meta['series_id']) & (group_diag_euclid_df['instance_number'] == meta['instance_number']) & (group_diag_euclid_df['condition'] == meta['condition'])]
        if group_row.empty:
            reports.append({'anchor_idx': anchor_idx, 'status': 'skipped_no_group_diag_row'})
            continue
        median_dist = float(group_row.iloc[0]['median_dist_px'])
        points = {}
        for _, r in anchor_df.iterrows():
            lvl_idx = int(r['level_idx'])
            points[lvl_idx] = np.array([float(r['pred_x']), float(r['pred_y'])], dtype=float)
        bad_upper_indices = []
        pair_diagnostics = []
        for upper_idx in [0, 1, 2, 3]:
            lower_idx = upper_idx + 1
            upper_level = LEVELS[upper_idx]
            lower_level = LEVELS[lower_idx]
            upper_xy = points[upper_idx]
            lower_xy = points[lower_idx]
            ratio_min, ratio_max = get_ratio_bounds(euclidean_ratio_constraints, from_level=upper_level, to_level=lower_level)
            min_allowed = median_dist * ratio_min
            max_allowed = median_dist * ratio_max
            current_dist = euclid_xy(upper_xy, lower_xy)
            order_ok = is_above(upper_xy, lower_xy, tol=tol)
            if not order_ok:
                diagnosis = 'wrong_order'
                is_bad_pair = True
            elif current_dist < min_allowed:
                diagnosis = 'too_close'
                is_bad_pair = True
            elif current_dist > max_allowed:
                diagnosis = 'too_distant'
                is_bad_pair = True
            else:
                diagnosis = 'valid'
                is_bad_pair = False
            pair_diagnostics.append({'pair': f'{upper_level}->{lower_level}', 'upper_idx_to_correct': upper_idx, 'upper_level': upper_level, 'lower_idx': lower_idx, 'lower_level': lower_level, 'diagnosis': diagnosis, 'is_bad_pair': is_bad_pair, 'current_dist': current_dist, 'median_dist_px': median_dist, 'ratio_min': ratio_min, 'ratio_max': ratio_max, 'min_allowed': min_allowed, 'max_allowed': max_allowed, 'order_ok': order_ok})
            if is_bad_pair:
                bad_upper_indices.append(upper_idx)
        if len(bad_upper_indices) == 0:
            reports.append({'anchor_idx': anchor_idx, 'status': 'no_bad_adjacent_pair_found', 'median_dist_px': median_dist, 'pair_diagnostics': pair_diagnostics})
            continue
        critical_idx = int(max(bad_upper_indices))
        correction_path = list(range(critical_idx, -1, -1))
        try:
            union_hmap = get_union_raw_heatmap_for_anchor(raw_heatmaps=raw_heatmaps, anchor_idx=anchor_idx, level_indices=[0, 1, 2, 3, 4], mode=union_mode)
        except KeyError:
            reports.append({'anchor_idx': anchor_idx, 'status': 'skipped_missing_union_raw_heatmap'})
            continue
        reports.append({'anchor_idx': anchor_idx, 'status': 'started_case', 'critical_idx': critical_idx, 'critical_level': LEVELS[critical_idx], 'lower_anchor_idx': critical_idx + 1, 'lower_anchor_level': LEVELS[critical_idx + 1], 'median_dist_px': median_dist, 'correction_path': [LEVELS[i] for i in correction_path], 'bad_upper_indices': bad_upper_indices, 'pair_diagnostics': pair_diagnostics})
        for upper_idx in correction_path:
            lower_idx = upper_idx + 1
            upper_level = LEVELS[upper_idx]
            lower_level = LEVELS[lower_idx]
            current_xy = points[upper_idx]
            lower_xy = points[lower_idx]
            ratio_min, ratio_max = get_ratio_bounds(euclidean_ratio_constraints, from_level=upper_level, to_level=lower_level)
            min_allowed = median_dist * ratio_min
            max_allowed = median_dist * ratio_max
            current_dist = euclid_xy(current_xy, lower_xy)
            if not is_above(current_xy, lower_xy, tol=tol):
                diagnosis = 'wrong_order'
            elif current_dist < min_allowed:
                diagnosis = 'too_close'
            elif current_dist > max_allowed:
                diagnosis = 'too_distant'
            else:
                diagnosis = 'valid'
            if diagnosis == 'valid':
                reports.append({'anchor_idx': anchor_idx, 'level_idx': upper_idx, 'level': upper_level, 'status': 'kept_valid', 'diagnosis': diagnosis, 'current_dist': current_dist, 'median_dist_px': median_dist, 'ratio_min': ratio_min, 'ratio_max': ratio_max, 'min_allowed': min_allowed, 'max_allowed': max_allowed})
                continue
            peaks = top_peaks_in_distance_range(hmap=union_hmap, lower_xy=lower_xy, min_allowed=min_allowed, max_allowed=max_allowed, top_k=top_k, nms_radius=nms_radius, min_conf=None)
            best = None
            best_score = -np.inf
            for p in peaks:
                cand_xy = np.array([p['x'], p['y']], dtype=float)
                cand_dist = float(p['candidate_dist'])
                dist_score = math.exp(-abs(cand_dist - median_dist) / median_dist)
                conf_score = float(p['confidence'])
                final_score = 0.65 * conf_score + 0.35 * dist_score
                if final_score > best_score:
                    best_score = final_score
                    best = {'xy': cand_xy, 'confidence': conf_score, 'candidate_dist': cand_dist, 'distance_score': dist_score, 'final_score': final_score}
            if best is None or best['final_score'] < min_candidate_score:
                reports.append({'anchor_idx': anchor_idx, 'level_idx': upper_idx, 'level': upper_level, 'status': f'reported_{diagnosis}_no_union_candidate', 'diagnosis': diagnosis, 'old_x': float(current_xy[0]), 'old_y': float(current_xy[1]), 'current_dist': current_dist, 'median_dist_px': median_dist, 'ratio_min': ratio_min, 'ratio_max': ratio_max, 'min_allowed': min_allowed, 'max_allowed': max_allowed, 'n_candidates_checked': len(peaks)})
                continue
            new_xy = best['xy']
            points[upper_idx] = new_xy
            corrected_df.loc[(corrected_df['anchor_idx'] == anchor_idx) & (corrected_df['level_idx'] == upper_idx), ['pred_x', 'pred_y']] = [float(new_xy[0]), float(new_xy[1])]
            reports.append({'anchor_idx': anchor_idx, 'level_idx': upper_idx, 'level': upper_level, 'status': f'auto_corrected_{diagnosis}', 'diagnosis': diagnosis, 'old_x': float(current_xy[0]), 'old_y': float(current_xy[1]), 'new_x': float(new_xy[0]), 'new_y': float(new_xy[1]), 'old_dist': current_dist, 'new_dist': best['candidate_dist'], 'median_dist_px': median_dist, 'ratio_min': ratio_min, 'ratio_max': ratio_max, 'min_allowed': min_allowed, 'max_allowed': max_allowed, 'candidate_confidence': best['confidence'], 'candidate_distance_score': best['distance_score'], 'candidate_final_score': best['final_score'], 'n_candidates_checked': len(peaks), 'candidate_source': 'union_raw_heatmap_all5'})
    return (corrected_df, pd.DataFrame(reports))


### Original code cell 73

In [ ]:
LEVEL_COLORS = {'L1/L2': 'cyan', 'L2/L3': 'yellow', 'L3/L4': 'lime', 'L4/L5': 'orange', 'L5/S1': 'red'}

def get_display_image_from_dataset(dataset, anchor_idx):
    """
    Loads the already-preprocessed image from test_ds[anchor_idx].
    Handles [C,H,W], [H,W], torch tensors, numpy arrays.
    """
    sample = dataset[int(anchor_idx)]
    image = sample['image']
    if hasattr(image, 'detach'):
        image = image.detach().cpu().numpy()
    image = np.asarray(image)
    if image.ndim == 3:
        c = image.shape[0] // 2
        image = image[c]
    image = image.astype(float)
    image = image - np.nanmin(image)
    denom = np.nanmax(image)
    if denom > 0:
        image = image / denom
    return image

def plot_original_vs_corrected_predictions_by_anchor(anchor_idx, original_pred_df, corrected_pred_df, dataset, figsize, marker_size):
    anchor_idx = int(anchor_idx)
    orig = original_pred_df[original_pred_df['anchor_idx'] == anchor_idx].sort_values('level_idx').copy()
    corr = corrected_pred_df[corrected_pred_df['anchor_idx'] == anchor_idx].sort_values('level_idx').copy()
    if orig.empty:
        raise ValueError(f'No original predictions found for anchor_idx={anchor_idx}')
    if corr.empty:
        raise ValueError(f'No corrected predictions found for anchor_idx={anchor_idx}')
    img = get_display_image_from_dataset(dataset, anchor_idx)
    meta = orig.iloc[0]
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    for ax, df_plot, title in [(axes[0], orig, 'Original prediction'), (axes[1], corr, 'Corrected prediction')]:
        ax.imshow(img, cmap='gray')
        for _, r in df_plot.iterrows():
            level = r['level']
            x = float(r['pred_x'])
            y = float(r['pred_y'])
            color = LEVEL_COLORS.get(level, 'cyan')
            ax.scatter(x, y, s=marker_size, c=color, marker='x', linewidths=2.5)
            ax.text(x + 3, y - 3, level, color=color, fontsize=11, fontweight='bold')
        ax.set_title(title)
        ax.axis('off')
    fig.suptitle(f"anchor_idx={anchor_idx} | study_id={meta.get('study_id')} | series_id={meta.get('series_id')} | instance_number={meta.get('instance_number')} | condition={meta.get('condition')}", fontsize=14)
    plt.tight_layout()
    plt.show()


### Original code cell 79

In [ ]:
def evaluate_full_test_with_corrected_coords(
    test_eval_df,
    corrected_all5_pred_df,
    threshold=EVAL_TOLERANCE_PX,
):
    key_cols = [
        "study_id",
        "series_id",
        "instance_number",
        "condition",
        "level",
    ]

    corrected_xy = corrected_all5_pred_df[
        key_cols + ["pred_x", "pred_y"]
    ].copy()

    corrected_xy = corrected_xy.rename(columns={
        "pred_x": "corrected_pred_x",
        "pred_y": "corrected_pred_y",
    })

    dupes = corrected_xy[corrected_xy.duplicated(key_cols, keep=False)]
    if len(dupes) > 0:
        raise ValueError(
            "corrected_all5_pred_df has duplicate rows for the merge key. "
            "Inspect `dupes`."
        )

    full_eval = test_eval_df.merge(
        corrected_xy,
        on=key_cols,
        how="left",
        validate="many_to_one",
    )

    full_eval["final_pred_x"] = full_eval["corrected_pred_x"].fillna(full_eval["pred_x"])
    full_eval["final_pred_y"] = full_eval["corrected_pred_y"].fillna(full_eval["pred_y"])

    full_eval["final_dist_px"] = np.sqrt(
        (full_eval["final_pred_x"] - full_eval["gt_x"]) ** 2
        + (full_eval["final_pred_y"] - full_eval["gt_y"]) ** 2
    )

    full_eval["final_within_10px"] = full_eval["final_dist_px"] <= threshold
    full_eval["final_outside_10px"] = full_eval["final_dist_px"] > threshold

    overall_summary_df = pd.DataFrame([{
        "n_predictions": len(full_eval),

        "original_correct_within_10px": int(full_eval["within_10px"].sum()),
        "original_incorrect_outside_10px": int(full_eval["outside_10px"].sum()),
        "original_hit_rate": float(full_eval["within_10px"].mean()),
        "original_mean_dist_px": float(full_eval["dist_px"].mean()),
        "original_median_dist_px": float(full_eval["dist_px"].median()),

        "final_correct_within_10px": int(full_eval["final_within_10px"].sum()),
        "final_incorrect_outside_10px": int(full_eval["final_outside_10px"].sum()),
        "final_hit_rate": float(full_eval["final_within_10px"].mean()),
        "final_mean_dist_px": float(full_eval["final_dist_px"].mean()),
        "final_median_dist_px": float(full_eval["final_dist_px"].median()),

        "fixed_bad_to_good": int(
            (full_eval["outside_10px"] & full_eval["final_within_10px"]).sum()
        ),
        "broken_good_to_bad": int(
            (full_eval["within_10px"] & full_eval["final_outside_10px"]).sum()
        ),
        "still_bad": int(
            (full_eval["outside_10px"] & full_eval["final_outside_10px"]).sum()
        ),
        "still_good": int(
            (full_eval["within_10px"] & full_eval["final_within_10px"]).sum()
        ),
    }])

    level_summary_df = (
        full_eval
        .groupby("level")
        .agg(
            n=("final_dist_px", "size"),

            original_within_10px=("within_10px", "sum"),
            original_hit_rate=("within_10px", "mean"),
            original_mean_dist=("dist_px", "mean"),
            original_median_dist=("dist_px", "median"),

            final_within_10px=("final_within_10px", "sum"),
            final_hit_rate=("final_within_10px", "mean"),
            final_mean_dist=("final_dist_px", "mean"),
            final_median_dist=("final_dist_px", "median"),
        )
    )

    level_summary_df["fixed_gain"] = (
        level_summary_df["final_within_10px"]
        - level_summary_df["original_within_10px"]
    )

    level_summary_df["hit_rate_gain"] = (
        level_summary_df["final_hit_rate"]
        - level_summary_df["original_hit_rate"]
    )

    return full_eval, overall_summary_df, level_summary_df


In [ ]:
def to_numpy_image(img):
    """
    Converts torch / numpy image to displayable 2D numpy image.
    Handles shapes:
    - H x W
    - 1 x H x W
    - C x H x W
    - H x W x C
    """
    if hasattr(img, "detach"):
        img = img.detach().cpu().numpy()

    img = np.asarray(img)

    if img.ndim == 3:
        # C x H x W
        if img.shape[0] in [1, 3, 5]:
            img = img[0]
        # H x W x C
        elif img.shape[-1] in [1, 3, 5]:
            img = img[..., 0]

    img = img.astype(np.float32)
    img = img - np.nanmin(img)
    img = img / (np.nanmax(img) + 1e-6)

    return img


def plot_prediction_misses(eval_df, dataset, ncols=4, max_plots=None):
    misses = eval_df[eval_df["dist_px"] > TOL].copy()
    misses = misses.sort_values("dist_px", ascending=False)

    if max_plots is not None:
        misses = misses.head(max_plots)

    n = len(misses)
    if n == 0:
        print(f"No predictions outside {int(TOL)}px.")
        return

    nrows = math.ceil(n / ncols)

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(4.2 * ncols, 4.2 * nrows)
    )

    axes = np.array(axes).reshape(-1)

    for ax, (_, row) in zip(axes, misses.iterrows()):
        sample = dataset[int(row["idx"])]

        # Try common image keys
        if "image" in sample:
            img = sample["image"]
        elif "img" in sample:
            img = sample["img"]
        elif "x" in sample:
            img = sample["x"]
        else:
            raise KeyError(
                f"Could not find image key in sample. Available keys: {list(sample.keys())}"
            )

        img = to_numpy_image(img)

        ax.imshow(img, cmap="gray")

        # GT point
        ax.scatter(
            row["gt_x"],
            row["gt_y"],
            s=70,
            c="lime",
            marker="x",
            linewidths=2,
            label="GT"
        )

        # Predicted point
        ax.scatter(
            row["pred_x"],
            row["pred_y"],
            s=70,
            c="red",
            marker="+",
            linewidths=2,
            label="Pred"
        )

        # Line between them
        ax.plot(
            [row["gt_x"], row["pred_x"]],
            [row["gt_y"], row["pred_y"]],
            color="yellow",
            linewidth=1.5,
            alpha=0.8
        )

        ax.set_title(
            f"idx={int(row['idx'])} | {row['level']}\n"
            f"dist={row['dist_px']:.1f}px | conf={row['raw_conf']:.3f}",
            fontsize=9
        )

        ax.axis("off")

    for ax in axes[n:]:
        ax.axis("off")

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper right")

    fig.suptitle(
        f"Predictions outside {int(TOL)} px tolerance",
        fontsize=14,
        y=1.01
    )

    plt.tight_layout()
    plt.show()
